In [ ]:
pip install dwave-ocean-sdk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.9/158.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 947.8 kB/s eta 0:00:00


## Crear la cuenta e importar las credenciales

In [ ]:
!dwave config create

Using the simplified configuration flow.
Try 'dwave config create --full' for more options.

Creating new configuration file: /root/.config/dwave/dwave.conf
Updating existing profile: defaults
Solver API token [skip]: DEV-8230fef3606ddf65bb468c190a2abd1aa9f883d0
Configuration saved.


## Trabajando con BLPs

In [ ]:
import dimod

x0 = dimod.Binary("x0")
x1 = dimod.Binary("x1")
x2 = dimod.Binary("x2")

blp = dimod.ConstrainedQuadraticModel()
blp.set_objective(-5*x0+3*x1-2*x2)
blp.add_constraint(x0 + x2 <= 1)
blp.add_constraint(3*x0 -x1 + 3*x2 <= 4)

qubo, invert = dimod.cqm_to_bqm(blp, lagrange_multiplier = 5)

In [ ]:
from dwave.system import DWaveSampler
from dwave.system import EmbeddingComposite

sampler = EmbeddingComposite(DWaveSampler())
result = sampler.sample(qubo, num_reads=100)

samples = []
occurrences = []
for s in result.data():
    samples.append(invert(s.sample))
    occurrences.append(s.num_occurrences)

sampleset = dimod.SampleSet.from_samples_cqm(samples,blp,
    num_occurrences=occurrences)
final_sols = sampleset.filter(lambda s: s.is_feasible)
final_sols = final_sols.aggregate()

print("Las soluciones obtenidas son:")
print(final_sols)

Las soluciones obtenidas son:
  x0 x1 x2 energy num_oc. is_sat. is_fea.
0  1  0  0   -5.0      32 arra...    True
1  0  0  1   -2.0      14 arra...    True
2  1  1  0   -2.0      30 arra...    True
3  0  0  0    0.0      11 arra...    True
4  0  1  1    1.0      10 arra...    True
5  0  1  0    3.0       2 arra...    True
['INTEGER', 6 rows, 99 samples, 3 variables]


## Explorando los quantum annealers

In [ ]:
from dwave.cloud import Client
for solver in Client.from_config().get_solvers():
    print(solver)

BQMSolver(id='hybrid_binary_quadratic_model_version2')
DQMSolver(id='hybrid_discrete_quadratic_model_version1')
CQMSolver(id='hybrid_constrained_quadratic_model_version1')
NLSolver(id='hybrid_nonlinear_program_version1')
StructuredSolver(id='Advantage2_prototype2.5')
StructuredSolver(id='Advantage_system6.4')
StructuredSolver(id='Advantage_system4.1')


In [ ]:
from dwave.system import DWaveSampler
sampler=DWaveSampler(solver='Advantage_system6.4')
print("Name:",sampler.properties["chip_id"])
print("Number of qubits:",sampler.properties["num_qubits"])
print("Category:",sampler.properties["category"])
print("Supported problems:",sampler.properties["supported_problem_types"])
print("Topology:",sampler.properties["topology"])
print("Range of reads:",sampler.properties["num_reads_range"])

Name: Advantage_system6.4
Number of qubits: 5760
Category: qpu
Supported problems: ['ising', 'qubo']
Topology: {'type': 'pegasus', 'shape': [16]}
Range of reads: [1, 10000]


In [ ]:
sampler=DWaveSampler(solver='Advantage_system4.1')
print("Name:",sampler.properties["chip_id"])
print("Number of qubits:",sampler.properties["num_qubits"])
print("Category:",sampler.properties["category"])
print("Supported problems:",sampler.properties["supported_problem_types"])
print("Topology:",sampler.properties["topology"])
print("Range of reads:",sampler.properties["num_reads_range"])

Name: Advantage_system4.1
Number of qubits: 5760
Category: qpu
Supported problems: ['ising', 'qubo']
Topology: {'type': 'pegasus', 'shape': [16]}
Range of reads: [1, 10000]


In [ ]:
sampler=DWaveSampler(solver='Advantage2_prototype2.5')
print("Name:",sampler.properties["chip_id"])
print("Number of qubits:",sampler.properties["num_qubits"])
print("Category:",sampler.properties["category"])
print("Supported problems:",sampler.properties["supported_problem_types"])
print("Topology:",sampler.properties["topology"])
print("Range of reads:",sampler.properties["num_reads_range"])

Name: Advantage2_prototype2.5
Number of qubits: 1248
Category: qpu
Supported problems: ['ising', 'qubo']
Topology: {'type': 'zephyr', 'shape': [6, 4]}
Range of reads: [1, 10000]


In [ ]:
sampler=DWaveSampler(solver='Advantage2_prototype2.5')
print("Couplings:",sampler.properties["couplers"])

Couplings: [[0, 1], [1, 2], [2, 3], [3, 4], [4, 5], [0, 6], [1, 6], [1, 7], [2, 7], [6, 7], [2, 8], [3, 8], [7, 8], [3, 9], [4, 9], [8, 9], [4, 10], [5, 10], [9, 10], [5, 11], [10, 11], [12, 13], [13, 14], [14, 15], [15, 16], [16, 17], [12, 18], [13, 18], [13, 19], [14, 19], [18, 19], [14, 20], [15, 20], [19, 20], [15, 21], [16, 21], [20, 21], [17, 23], [27, 28], [28, 29], [25, 30], [25, 31], [30, 31], [27, 32], [31, 32], [27, 33], [28, 33], [32, 33], [28, 34], [29, 34], [33, 34], [29, 35], [34, 35], [36, 37], [37, 38], [38, 39], [39, 40], [40, 41], [36, 42], [37, 42], [37, 43], [38, 43], [42, 43], [38, 44], [39, 44], [43, 44], [39, 45], [40, 45], [44, 45], [40, 46], [41, 46], [45, 46], [48, 49], [49, 50], [50, 51], [51, 52], [52, 53], [48, 54], [49, 54], [49, 55], [50, 55], [54, 55], [50, 56], [51, 56], [55, 56], [51, 57], [52, 57], [56, 57], [52, 58], [53, 58], [57, 58], [53, 59], [58, 59], [60, 61], [64, 65], [60, 66], [61, 66], [61, 67], [66, 67], [67, 68], [64, 69], [68, 69], [64,

In [ ]:
sampler.adjacency

{0: {1, 6, 624, 636, 648, 660, 672, 684, 696, 708},
 1: {0, 2, 6, 7, 720, 732, 744, 756, 768, 780, 792, 804},
 2: {1, 3, 7, 8, 816, 828, 840, 852, 864, 876, 900},
 3: {2, 4, 8, 9, 912, 924, 936, 948, 960, 972, 984, 996},
 4: {3, 5, 9, 10, 1008, 1020, 1032, 1044, 1056, 1068, 1080, 1092},
 5: {4, 10, 11, 1104, 1116, 1128, 1140, 1152, 1164, 1176, 1188},
 6: {0, 1, 7, 672, 684, 696, 708, 720, 732, 744, 756},
 7: {1, 2, 6, 8, 768, 780, 792, 804, 816, 828, 840, 852},
 8: {2, 3, 7, 9, 864, 876, 900, 912, 924, 936, 948},
 9: {3, 4, 8, 10, 960, 972, 984, 996, 1008, 1020, 1032, 1044},
 10: {4, 5, 9, 11, 1056, 1068, 1080, 1092, 1104, 1116, 1128, 1140},
 11: {5, 10, 1152, 1164, 1176, 1188, 1200, 1212, 1224, 1236},
 12: {13, 18, 624, 636, 648, 660, 672, 684, 696, 708},
 13: {12, 14, 18, 19, 720, 732, 744, 756, 768, 780, 792, 804},
 14: {13, 15, 19, 20, 816, 828, 840, 852, 864, 876, 900},
 15: {14, 16, 20, 21, 912, 924, 936, 948, 960, 972, 984, 996},
 16: {15, 17, 21, 1008, 1020, 1032, 1044, 1056, 1

## Cambiando los parámetros de ejecución

In [ ]:
sampler = DWaveSampler(solver = "Advantage_system4.1")
print("El tiempo de annealing por defecto es",
    sampler.properties["default_annealing_time"],"microsegundos")
print("El rango del tiempo de annealing en microsegundos es"\
    ,sampler.properties["annealing_time_range"])

El tiempo de annealing por defecto es 20.0 microsegundos
El rango del tiempo de annealing en microsegundos es [0.5, 2000.0]


In [ ]:
J = {(0,1):1, (0,2):1, (1,2):1}
h = {}
triangle = dimod.BinaryQuadraticModel(h, J, 0.0, dimod.SPIN)
sampler = EmbeddingComposite(DWaveSampler(solver = "Advantage_system4.1"))
result = sampler.sample(triangle, num_reads=10, annealing_time = 100)
print("Los resultados obtenidos son:")
print(result)

Los resultados obtenidos son:
   0  1  2 energy num_oc. chain_.
0 -1 +1 +1   -1.0       1     0.0
1 +1 -1 -1   -1.0       3     0.0
2 +1 +1 -1   -1.0       1     0.0
3 -1 +1 -1   -1.0       2     0.0
4 +1 -1 +1   -1.0       3     0.0
['SPIN', 5 rows, 10 samples, 3 variables]


In [ ]:
forward_schedule=[[0.0, 0.0], [5.0, 0.25], [25, 0.75], [30, 1.0]]
sampler = EmbeddingComposite(DWaveSampler())
result = sampler.sample(triangle, num_reads=10,
    anneal_schedule = forward_schedule)
print("Los resultados obtenidos son:")
print(result)

Los resultados obtenidos son:
   0  1  2 energy num_oc. chain_.
0 +1 -1 +1   -1.0       1     0.0
1 -1 -1 +1   -1.0       2     0.0
2 +1 -1 -1   -1.0       1     0.0
3 -1 +1 -1   -1.0       2     0.0
4 +1 +1 -1   -1.0       2     0.0
5 -1 +1 +1   -1.0       2     0.0
['SPIN', 6 rows, 10 samples, 3 variables]


In [ ]:
reverse_schedule=[[0.0, 1.0], [10.0, 0.5], [20, 1.0]]
initial_state = {0:1, 1:1, 2:1}
sampler = EmbeddingComposite(DWaveSampler())
result = sampler.sample(triangle, num_reads=10,
    anneal_schedule = reverse_schedule,
    reinitialize_state=False, initial_state = initial_state)
print("TLos resultados obtenidos son:")
print(result)

TLos resultados obtenidos son:
   0  1  2 energy num_oc. chain_.
0 +1 -1 +1   -1.0       1     0.0
1 +1 -1 -1   -1.0       1     0.0
2 +1 -1 +1   -1.0       1     0.0
3 -1 +1 +1   -1.0       1     0.0
4 +1 -1 +1   -1.0       1     0.0
5 +1 +1 -1   -1.0       1     0.0
6 +1 -1 -1   -1.0       1     0.0
7 -1 +1 -1   -1.0       1     0.0
8 +1 -1 +1   -1.0       1     0.0
9 -1 +1 +1   -1.0       1     0.0
['SPIN', 10 rows, 10 samples, 3 variables]


## La importancia del rango de los coeficientes

In [ ]:
sampler = EmbeddingComposite(DWaveSampler("Advantage2_prototype2.5"))

x0 = dimod.Binary("x0")
x1 = dimod.Binary("x1")
x2 = dimod.Binary("x2")
blp = dimod.ConstrainedQuadraticModel()
blp.set_objective(-5*x0+3*x1-2*x2)
blp.add_constraint(x0 + x2 <= 1)
blp.add_constraint(3*x0 -x1 + 3*x2 <= 4)

qubo, invert = dimod.cqm_to_bqm(blp, lagrange_multiplier = 10)
result = sampler.sample(qubo, num_reads=100)

samples = []
occurrences = []
for s in result.data():
    samples.append(invert(s.sample))
    occurrences.append(s.num_occurrences)
sampleset = dimod.SampleSet.from_samples_cqm(samples,blp,
    num_occurrences=occurrences)
print("Las soluciones obtenidas son: ")
print(sampleset.filter(lambda s: s.is_feasible).aggregate())

Las soluciones obtenidas son: 
  x0 x1 x2 energy num_oc. is_sat. is_fea.
0  1  0  0   -5.0      16 arra...    True
1  1  1  0   -2.0      29 arra...    True
2  0  0  1   -2.0      14 arra...    True
3  0  0  0    0.0       5 arra...    True
4  0  1  1    1.0      31 arra...    True
5  0  1  0    3.0       4 arra...    True
['INTEGER', 6 rows, 99 samples, 3 variables]


In [ ]:
qubo, invert = dimod.cqm_to_bqm(blp, lagrange_multiplier = 4)
result = sampler.sample(qubo, num_reads=100)

samples = []
occurrences = []
for s in result.data():
    samples.append(invert(s.sample))
    occurrences.append(s.num_occurrences)
sampleset = dimod.SampleSet.from_samples_cqm(samples,blp,
    num_occurrences=occurrences)
print("Las soluciones obtenidas son: ")
print(sampleset.filter(lambda s: s.is_feasible).aggregate())

Las soluciones obtenidas son: 
  x0 x1 x2 energy num_oc. is_sat. is_fea.
0  1  0  0   -5.0      38 arra...    True
1  1  1  0   -2.0      18 arra...    True
2  0  0  1   -2.0      14 arra...    True
3  0  0  0    0.0       9 arra...    True
4  0  1  1    1.0      17 arra...    True
5  0  1  0    3.0       4 arra...    True
['INTEGER', 6 rows, 100 samples, 3 variables]


In [ ]:
qubo, invert = dimod.cqm_to_bqm(blp, lagrange_multiplier = 1)
result = sampler.sample(qubo, num_reads=100)

samples = []
occurrences = []
for s in result.data():
    samples.append(invert(s.sample))
    occurrences.append(s.num_occurrences)
sampleset = dimod.SampleSet.from_samples_cqm(samples,blp,
    num_occurrences=occurrences)
print("Las soluciones obtenidas son: ")
print(sampleset.filter(lambda s: s.is_feasible).aggregate())

Las soluciones obtenidas son: 
  x0 x1 x2 energy num_oc. is_sat. is_fea.
0  1  0  0   -5.0      79 arra...    True
1  1  1  0   -2.0      11 arra...    True
2  0  0  1   -2.0       5 arra...    True
3  0  1  1    1.0       1 arra...    True
4  0  1  0    3.0       1 arra...    True
['INTEGER', 5 rows, 97 samples, 3 variables]


## Métodos clásicos e híbridos

In [ ]:
import greedy

J = {(0,1):1, (1,2):1, (2,3):1, (3,0):1}
h = {}
problem = dimod.BinaryQuadraticModel(h, J, 0.0, dimod.SPIN)

solver = greedy.SteepestDescentSolver()
solution = solver.sample(problem, num_reads = 10)
print(solution.aggregate())

   0  1  2  3 energy num_oc. num_st.
1 +1 -1 +1 -1   -4.0       2       1
2 -1 +1 -1 +1   -4.0       7       1
0 +1 +1 -1 -1    0.0       1       0
['SPIN', 3 rows, 10 samples, 4 variables]


In [ ]:
import tabu
solver = tabu.TabuSampler()
solution = solver.sample(problem, num_reads = 10)
print(solution.aggregate())

   0  1  2  3 energy num_oc. num_re.
0 -1 +1 -1 +1   -4.0       6       1
1 +1 -1 +1 -1   -4.0       4       1
['SPIN', 2 rows, 10 samples, 4 variables]


In [ ]:
import neal
solver = neal.SimulatedAnnealingSampler()
solution = solver.sample(problem, num_reads = 10)
print(solution.aggregate())

   0  1  2  3 energy num_oc.
2 +1 -1 +1 -1   -4.0       2
4 -1 +1 -1 +1   -4.0       1
0 +1 +1 +1 -1    0.0       3
1 +1 -1 -1 -1    0.0       2
3 -1 -1 +1 +1    0.0       2
['SPIN', 5 rows, 10 samples, 4 variables]


In [ ]:
import dwave.system
sampler = dwave.system.LeapHybridSampler()
solution = solver.sample(problem, num_reads = 10)
print(solution.aggregate())

   0  1  2  3 energy num_oc.
0 +1 -1 +1 -1   -4.0       4
1 -1 +1 -1 +1   -4.0       1
2 -1 -1 -1 +1    0.0       2
3 +1 +1 -1 -1    0.0       1
4 -1 +1 +1 +1    0.0       2
['SPIN', 5 rows, 10 samples, 4 variables]


In [ ]:
sampler.properties["quota_conversion_rate"]


20